# Gemma-2 / Bhojpuri headline — EVALUATION ONLY

Your 218 generations are already on disk. This notebook **reads** them and runs
the evaluation; it never loads a model and never regenerates.

**Do not re-run `run_gemma2_bho_headline.ipynb`** — its Cell 1 sets
`FRESH_START = True`, so Cell 8 would move your checkpoint aside and start over.

Cell 1 makes a backup copy of the checkpoint before doing anything else.

Run top to bottom. No GPU needed (BERTScore runs on CPU for 218 items in a
couple of minutes).


In [1]:
# ============================================================
# CELL 1 — locate and BACK UP the existing generations
# ============================================================
import os, glob, shutil, time, json
import numpy as np, pandas as pd

RUN_TAG   = "gemma2_bho_headline_zeroshot"
TEST_CSV  = "bho_test.csv"
EXPECT_N  = 218

CKPT  = f"ckpt_{RUN_TAG}.csv"
FINAL = f"gen_{RUN_TAG}.csv"
PERITEM_CSV   = f"per_item_{RUN_TAG}.csv"
METRICS_JSON  = f"metrics_{RUN_TAG}.json"
MANIFEST_JSON = f"manifest_{RUN_TAG}.json"

print("files present:")
for p in (TEST_CSV, CKPT, FINAL):
    print(f"   {p:<45} {'yes' if os.path.exists(p) else 'NO'}")

src = FINAL if os.path.exists(FINAL) else CKPT
if not os.path.exists(src):
    raise SystemExit(f"No generations found. Looked for {FINAL} and {CKPT} in {os.getcwd()}")

# back up before touching anything
bak = f"{src}.backup.{int(time.time())}"
shutil.copy2(src, bak)
print(f"\nusing   : {src}")
print(f"backup  : {bak}")


files present:
   bho_test.csv                                  yes
   ckpt_gemma2_bho_headline_zeroshot.csv         yes
   gen_gemma2_bho_headline_zeroshot.csv          yes

using   : gen_gemma2_bho_headline_zeroshot.csv
backup  : gen_gemma2_bho_headline_zeroshot.csv.backup.1790505290


In [2]:
# ============================================================
# CELL 2 — rebuild the dataframe: test split + saved generations
# ============================================================
TEXT_COL, REF_COL, GEN_COL = "article_text", "headline", "generated"

t = pd.read_csv(TEST_CSV, dtype=str, keep_default_na=False)
g = pd.read_csv(src,      dtype=str, keep_default_na=False)
print(f"test split {len(t)} rows | saved generations {len(g)} rows")
assert len(t) == EXPECT_N, f"{TEST_CSV} has {len(t)} rows, expected {EXPECT_N} — wrong file"

gen_col = GEN_COL if GEN_COL in g.columns else [c for c in g.columns if c.startswith("generated")][0]
m = dict(zip(g["row_key"].astype(str), g[gen_col].astype(str)))

df = t.copy()
df[GEN_COL] = df["row_key"].astype(str).map(lambda k: m.get(k, ""))

matched   = int(df["row_key"].astype(str).isin(m).sum())
n_empty   = int((df[GEN_COL].str.strip() == "").sum())
print(f"matched by row_key : {matched}/{EXPECT_N}")
print(f"empty generations  : {n_empty}")
if matched < EXPECT_N:
    print("\n!! Some test rows have no generation. If this is a partial checkpoint,")
    print("   re-run the original notebook with FRESH_START = False to finish the")
    print("   missing rows only, then come back here.")

w = df[GEN_COL].str.split().str.len()
print(f"\ngenerated length: mean {w.mean():.2f} med {w.median():.0f} "
      f"std {w.std(ddof=1):.2f} >10w {100*(w>10).mean():.1f}%")
print("\n--- 3 samples ---")
for i in df.index[:3]:
    print(f"GOLD: {df.at[i, REF_COL][:85]}")
    print(f"GEN : {df.at[i, GEN_COL][:85]}\n")


test split 218 rows | saved generations 218 rows
matched by row_key : 218/218
empty generations  : 0

generated length: mean 17.33 med 18 std 9.23 >10w 61.9%

--- 3 samples ---
GOLD: South Korea: प्लेन क्रैश में 85 से जादे यत्रियन के मौत: 181 लोग सवार रहे, बाकियो के म
GEN : इंग ना हो सकेल। **शीर्ष्क:** जेजू एयर प्‍लेन मूआन एअरपोर्ट में क्रश

GOLD: Champions Trophy: भारत के फाइनल में पहुंचला पs पाकिस्तान में ना होई खिताबी मुकाबला, र
GEN : मीन पर खिताब जीतल बिया, तs भारत के खिलाड़ियों के सुरक्षा के खतरा रहे। **उदाहरण शीर्षक

GOLD: भारतीय स्ट्राइकर वंदना कटारिया इंटरनेशनल हॉकी के कहली अलविदा
GEN : परंतु खेल के साथी, प्रशंसक आ दोस्तों के साथ उs हमेशा जुड़े रहलहे। वंदना काटेरिया हॉके 



In [3]:
# ============================================================
# CELL 3 — STANDARDIZED EVALUATION  (same as Cell 9 of the run notebooks)
#   ROUGE word-level per-instance F1 | BLEU sentence-level smoothed intl
#   BERTScore mBERT | empties scored 0, kept | bootstrap 95% CIs
# ============================================================
SEED = 42
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

class _WS:
    def tokenize(self, text): return str(text).split()
_sc = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"],
                               use_stemmer=False, tokenizer=_WS())

def _ci(v, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed); a = np.asarray(v, dtype=float)
    b = [a[rng.integers(0, len(a), len(a))].mean() for _ in range(n_boot)]
    return [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]

refs = df[REF_COL].fillna("").astype(str).str.strip().tolist()
hyps = df[GEN_COL].fillna("").astype(str).str.strip().tolist()
assert len(refs) == len(hyps) == EXPECT_N, "row count drift — do not report these numbers"

per = [_sc.score(r, h) for r, h in zip(refs, hyps)]      # empty hyp -> 0.0, kept
r1 = [p["rouge1"].fmeasure for p in per]
r2 = [p["rouge2"].fmeasure for p in per]
rl = [p["rougeL"].fmeasure for p in per]
bl = [sacrebleu.sentence_bleu(h, [r], smooth_method="exp", tokenize="intl").score/100.0
      for h, r in zip(hyps, refs)]
cb = sacrebleu.corpus_bleu(hyps, [refs], tokenize="intl")   # for brevity penalty

# bert-score / transformers version clash: restore the missing method
import transformers as _tf
for _cls in [_tf.BertTokenizer, getattr(_tf, "BertTokenizerFast", None)]:
    if _cls is not None and not hasattr(_cls, "build_inputs_with_special_tokens"):
        def _build_inputs(self, token_ids_0, token_ids_1=None):
            cls_id, sep_id = [self.cls_token_id], [self.sep_token_id]
            if token_ids_1 is None:
                return cls_id + token_ids_0 + sep_id
            return cls_id + token_ids_0 + sep_id + token_ids_1 + sep_id
        _cls.build_inputs_with_special_tokens = _build_inputs

_, _, F = bert_score_fn(hyps, refs, lang="hi",
                        model_type="bert-base-multilingual-cased",
                        batch_size=8, verbose=False)
bs   = F.tolist()
lens = [len(h.split()) for h in hyps]

M = {"run":RUN_TAG, "model":"google/gemma-2-9b-it", "lang":"bho", "task":"headline",
     "shots":0, "n":EXPECT_N,
     "n_empty":int(sum(1 for h in hyps if not h)),
     "valid_rate":float(np.mean([1.0 if h else 0.0 for h in hyps])),
     "bleu_sentence":float(np.mean(bl)), "bleu_sentence_ci":_ci(bl),
     "rouge1":float(np.mean(r1)), "rouge1_ci":_ci(r1),
     "rouge2":float(np.mean(r2)), "rouge2_ci":_ci(r2),
     "rougeL":float(np.mean(rl)), "rougeL_ci":_ci(rl),
     "bertscore_f1":float(np.mean(bs)), "bertscore_f1_ci":_ci(bs),
     "brevity_penalty":float(cb.bp), "bleu_corpus":float(cb.score/100.0),
     "len_mean":float(np.mean(lens)), "len_median":float(np.median(lens)),
     "len_std":float(np.std(lens, ddof=1)),
     "pct_over_10w":float(np.mean([l > 10 for l in lens]))}

print(f"\n{'='*66}\n{RUN_TAG}   n={M['n']}   valid={M['valid_rate']:.1%}   "
      f"empty={M['n_empty']}\n{'='*66}")
for k in ("bleu_sentence","rouge1","rouge2","rougeL","bertscore_f1"):
    lo, hi = M[k+"_ci"]
    print(f"  {k:<15} {M[k]:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
print(f"  brevity_penalty {M['brevity_penalty']:.4f}   (1.0 = no penalty)")
print(f"  length          mean={M['len_mean']:.2f} med={M['len_median']:.1f} "
      f"std={M['len_std']:.2f} >10w={M['pct_over_10w']:.1%}")

pd.DataFrame({"row_key":df["row_key"], "ref":refs, "hyp":hyps, "rouge1":r1,
              "rouge2":r2, "rougeL":rl, "bleu":bl, "bertscore":bs, "len":lens}
            ).to_csv(PERITEM_CSV, index=False, encoding="utf-8-sig")
json.dump(M, open(METRICS_JSON,"w"), indent=2, ensure_ascii=False)
print(f"\nsaved -> {PERITEM_CSV}\nsaved -> {METRICS_JSON}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



gemma2_bho_headline_zeroshot   n=218   valid=100.0%   empty=0
  bleu_sentence   0.0204   95% CI [0.0187, 0.0223]
  rouge1          0.1113   95% CI [0.1002, 0.1231]
  rouge2          0.0078   95% CI [0.0048, 0.0113]
  rougeL          0.0957   95% CI [0.0863, 0.1061]
  bertscore_f1    0.6749   95% CI [0.6699, 0.6805]
  brevity_penalty 1.0000   (1.0 = no penalty)
  length          mean=17.33 med=18.0 std=9.23 >10w=61.9%

saved -> per_item_gemma2_bho_headline_zeroshot.csv
saved -> metrics_gemma2_bho_headline_zeroshot.json


In [4]:
# ============================================================
# CELL 4 — protocol manifest (records the CORRECTED test set)
# ============================================================
manifest = {
 "run":RUN_TAG, "model_name":"google/gemma-2-9b-it", "architecture":"causal",
 "chat_format":"chat_nosys", "language":"bho", "task":"headline",
 "eval_set":TEST_CSV, "n_eval":EXPECT_N, "split_seed":42,
 "reference_column":REF_COL, "shots":0, "exemplar":None,
 "prompt_sha1":"cad4a5f2bac8e991", "affect_clause_in_prompt":True,
 "decoding":{"num_beams":4, "length_penalty":1.0, "no_repeat_ngram_size":3,
             "do_sample":False, "max_new_tokens":64, "max_input_tokens":1024},
 "precision":"4bit-nf4",
 "postprocessing":"marker + label cleanup only; NO length cap",
 "empty_output_policy":"scored as 0, never dropped",
 "metrics":{"rouge":"word-level whitespace, per-instance F1, averaged",
            "bleu":"sentence-level, smooth_method=exp, tokenize=intl, /100",
            "bertscore":"bert-base-multilingual-cased, lang=hi, no rescaling",
            "ci":"1000-sample bootstrap, 95% percentile"}}
json.dump(manifest, open(MANIFEST_JSON,"w"), indent=2, ensure_ascii=False)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

assert manifest["eval_set"] == "bho_test.csv" and manifest["n_eval"] == 218, \
    "manifest does not record the corrected test set"
print("\nOK: manifest records bho_test.csv with n=218 "
      "(the previous run recorded 'test (1).csv' with n=217)")


{
  "run": "gemma2_bho_headline_zeroshot",
  "model_name": "google/gemma-2-9b-it",
  "architecture": "causal",
  "chat_format": "chat_nosys",
  "language": "bho",
  "task": "headline",
  "eval_set": "bho_test.csv",
  "n_eval": 218,
  "split_seed": 42,
  "reference_column": "headline",
  "shots": 0,
  "exemplar": null,
  "prompt_sha1": "cad4a5f2bac8e991",
  "affect_clause_in_prompt": true,
  "decoding": {
    "num_beams": 4,
    "length_penalty": 1.0,
    "no_repeat_ngram_size": 3,
    "do_sample": false,
    "max_new_tokens": 64,
    "max_input_tokens": 1024
  },
  "precision": "4bit-nf4",
  "postprocessing": "marker + label cleanup only; NO length cap",
  "empty_output_policy": "scored as 0, never dropped",
  "metrics": {
    "rouge": "word-level whitespace, per-instance F1, averaged",
    "bleu": "sentence-level, smooth_method=exp, tokenize=intl, /100",
    "bertscore": "bert-base-multilingual-cased, lang=hi, no rescaling",
    "ci": "1000-sample bootstrap, 95% percentile"
  }
}

OK:

## After running

Send me `metrics_gemma2_bho_headline_zeroshot.json` and the manifest. The
ROUGE-1 value replaces 0.128 in Table 13, and the confidence interval goes into
the CI column.

If Cell 2 reports fewer than 218 matched rows, the checkpoint is partial. In that
case open the original notebook, set **`FRESH_START = False`** in Cell 1, and run
it — Cell 8 will generate only the missing rows and keep what you already have.
Then return to this notebook.
